In [0]:
# Libraries import

import mlflow
import pandas as pd
import numpy as np
from mlflow.tracking import MlflowClient

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

client   = MlflowClient()
username = spark.sql("SELECT current_user()").collect()[0][0]


def get_runs_from_experiment(experiment_path):
    """Fetch all runs from an MLflow experiment path."""
    experiment = mlflow.get_experiment_by_name(experiment_path)
    if experiment is None:
        print(f"Experiment not found: {experiment_path}")
        return pd.DataFrame()

    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"]
    )
    return runs

# Collecting runs from all three experiments
runs_supervised = get_runs_from_experiment(f"/Users/{username}/supervised_models")
runs_anomaly    = get_runs_from_experiment(f"/Users/{username}/anomaly_detection")
runs_lstm       = get_runs_from_experiment(f"/Users/{username}/lstm")

print(f"Supervised runs found  : {len(runs_supervised)}")
print(f"Anomaly runs found     : {len(runs_anomaly)}")
print(f"LSTM runs found        : {len(runs_lstm)}")

# Parsing results in a clean table

def parse_runs(runs_df, source="supervised"):
    """
    Extract model name, dataset, and key metrics from MLflow runs DataFrame.
    """
    if runs_df.empty:
        return []

    records = []
    for _, row in runs_df.iterrows():
        run_name = row.get("tags.mlflow.runName", "unknown")

        # This part skips nested runs
        if pd.isna(run_name) or run_name == "unknown":
            continue

        # Parse model name and dataset from run_name convention
        # Convention used: "LR_applications", "RF_transactions", "XGB_applications".
        parts   = run_name.split("_", 1)
        model   = parts[0] if len(parts) > 0 else run_name
        dataset = parts[1] if len(parts) > 1 else row.get("params.dataset", "unknown")

        record = {
            "run_name"      : run_name,
            "model"         : model,
            "dataset"       : dataset,
            "source"        : source,
            "roc_auc"       : row.get("metrics.roc_auc",        np.nan),
            "pr_auc"        : row.get("metrics.pr_auc",         np.nan),
            "f1"            : row.get("metrics.f1",             np.nan),
            "best_threshold": row.get("metrics.best_threshold", np.nan),
            "run_id"        : row.get("run_id", ""),
        }
        records.append(record)

    return records

all_records = (
    parse_runs(runs_supervised, "supervised") +
    parse_runs(runs_anomaly,    "anomaly")    +
    parse_runs(runs_lstm,       "lstm")
)

results_df = pd.DataFrame(all_records)

# Dropping duplicates, keeping most recent results

results_df = results_df.drop_duplicates(
    subset=["model", "dataset"],
    keep="first"
).reset_index(drop=True)

print(f"\nTotal unique model-dataset combinations: {len(results_df)}")


# Results for display

results_display = results_df[[
    "model", "dataset", "source",
    "roc_auc", "pr_auc", "f1", "best_threshold"
]].copy()

results_display = results_display.sort_values(
    ["dataset", "pr_auc"],
    ascending=[True, False]
)

# Format for readability
for col in ["roc_auc", "pr_auc", "f1", "best_threshold"]:
    results_display[col] = results_display[col].round(4)

print("\n" + "="*70)
print("  FULL PLATFORM MODEL COMPARISON")
print("="*70)
display(results_display)

# For enhancing focus analysis, results were divided by dataset
print("\n" + "="*70)
print("  APPLICATIONS — All Models")
print("="*70)
app_results = results_display[
    results_display["dataset"].str.contains("application", case=False, na=False)
].sort_values("pr_auc", ascending=False)
display(app_results)

print("\n" + "="*70)
print("  TRANSACTIONS — All Models")
print("="*70)
trans_results = results_display[
    results_display["dataset"].str.contains("transaction", case=False, na=False)
].sort_values("pr_auc", ascending=False)
display(trans_results)

#  Best model per dataset

print("\n" + "="*70)
print("  BEST MODEL PER DATASET (by PR-AUC)")
print("="*70)

for dataset in results_display["dataset"].unique():
    subset = results_display[results_display["dataset"] == dataset]
    best   = subset.loc[subset["pr_auc"].idxmax()]
    print(f"\n  {dataset.upper()}")
    print(f"    Best model     : {best['model']}")
    print(f"    ROC-AUC        : {best['roc_auc']:.4f}")
    print(f"    PR-AUC         : {best['pr_auc']:.4f}   ← key metric")
    print(f"    F1             : {best['f1']:.4f}")
    print(f"    Best threshold : {best['best_threshold']:.4f}")


# Final diagnostic to validate is any model is missing
expected_models = {
    "applications" : ["LR", "RF", "XGB", "Autoencoder", "IsolationForest"],
    "transactions" : ["LR", "RF", "XGB", "Autoencoder", "IsolationForest", "LSTM"]
}

print("\n" + "="*70)
print("  COVERAGE CHECK — Expected vs Found")
print("="*70)

for dataset, models in expected_models.items():
    subset       = results_display[results_display["dataset"].str.contains(dataset, case=False, na=False)]
    found_models = subset["model"].tolist()
    for m in models:
        found  = any(m.lower() in f.lower() for f in found_models)
        status = "✅" if found else "❌ MISSING"
        print(f"  {status}  {m} | {dataset}")